# AlphaEarth-Embeddings für Bayern – jahrweiser Export

Dieses Notebook exportiert **immer nur ein ausgewähltes Jahr** der AlphaEarth-Embeddings nach Google Drive.

Ein vollständiges Jahr für Bayern benötigt laut Earth Engine ungefähr **141–142 GB**. Bei 200 GB Drive-Speicher sollte daher jeweils nur ein Jahr exportiert werden:

1. Jahr auswählen.
2. Export vollständig abschließen lassen.
3. Dateien aus Google Drive lokal herunterladen.
4. Dateien anschließend aus Google Drive löschen und auch den Papierkorb leeren.
5. Nächstes Jahr auswählen und das Notebook erneut ausführen.

Die Ausgabe wird in mehrere GeoTIFF-Dateien aufgeteilt. Die Gesamtgröße wird dadurch nicht kleiner, der Download ist aber robuster als bei einer einzelnen sehr großen Datei.

In [ ]:
import sys

# Nur beim ersten Durchlauf beziehungsweise falls das Paket fehlt:
#!{sys.executable} -m pip install earthengine-api -q

In [1]:
import ee
import time
from datetime import datetime

print("Earth Engine API:", ee.__version__)

Earth Engine API: 1.7.35


In [ ]:
import ee

try:
    ee.Initialize()
except:
    ee.Authenticate()
    ee.Initialize()


In [2]:
import ee

PROJECT_ID = "embeddingstest-480211"

try:
    ee.Initialize(project=PROJECT_ID)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=PROJECT_ID)

In [3]:
for t in ee.batch.Task.list():
    s = t.status()

    if s.get("state") in {"READY", "RUNNING"}:
        print(
            s.get("description"),
            s.get("state"),
            s.get("creation_timestamp_ms")
        )

In [9]:
# stop running tasks

for t in ee.batch.Task.list():
    s = t.status()

    if (
        s.get("description") == "AlphaEarth_Bayern_2021"
        and s.get("state") in {"READY", "RUNNING"}
    ):
        t.cancel()
        print("Gestoppt:", s.get("state"), s.get("id"))

## 1. Earth Engine initialisieren

Falls dein Earth-Engine-Zugang an ein bestimmtes Google-Cloud-Projekt gebunden ist, trage dessen Projekt-ID bei `EE_PROJECT` ein.

In [ ]:
# # Optional: Earth-Engine-/Google-Cloud-Projekt eintragen.
# # Beispiel: EE_PROJECT = "mein-earth-engine-projekt"
# EE_PROJECT = None

# try:
#     if EE_PROJECT:
#         ee.Initialize(project=EE_PROJECT)
#     else:
#         ee.Initialize()
# except Exception:
#     ee.Authenticate()
#     if EE_PROJECT:
#         ee.Initialize(project=EE_PROJECT)
#     else:
#         ee.Initialize()

# print("Earth Engine initialized")

## 2. Exportjahr festlegen

Hier immer **genau ein Jahr** auswählen. Nach dem erfolgreichen Download und dem Löschen der Dateien aus Drive kannst du den Wert auf das nächste Jahr setzen.

Das Notebook prüft vor dem Export, ob für das gewählte Jahr Bilder vorhanden sind.

In [10]:
# ------------------------------------------------------------
# NUR HIER DAS GEWÜNSCHTE JAHR ÄNDERN
# ------------------------------------------------------------

YEAR = 2022

# Zielordner in Google Drive
DRIVE_FOLDER = "AlphaEarth_Bavaria"

# Dateipräfix
FILE_PREFIX = f"alphaearth_bayern_{YEAR}"

# 4096 x 4096 Pixel pro Ausgabedatei:
# bei 10 m ungefähr 40.96 x 40.96 km.
# Bei 64 float64-Bändern entspricht das etwa 8 GiB
# vor Kompression und bleibt unter dem Earth-Engine-Limit
# von 16 GiB pro Ausgabedatei.
FILE_DIMENSIONS = 4096

# Sicherheit: erwarteter freier Drive-Speicher
EXPECTED_FREE_DRIVE_GB = 200
ESTIMATED_EXPORT_GB = 142

if EXPECTED_FREE_DRIVE_GB < ESTIMATED_EXPORT_GB + 5:
    raise RuntimeError(
        "Voraussichtlich zu wenig freier Drive-Speicher. "
        "Für ein Jahr werden etwa 141–142 GB benötigt."
    )

print("Ausgewähltes Jahr:", YEAR)
print("Drive-Ordner:", DRIVE_FOLDER)
print("Dateipräfix:", FILE_PREFIX)

Ausgewähltes Jahr: 2022
Drive-Ordner: AlphaEarth_Bavaria
Dateipräfix: alphaearth_bayern_2022


## 3. Bayern-Grenze und AlphaEarth-Daten laden

In [11]:
bavaria = (
    ee.FeatureCollection("FAO/GAUL/2015/level1")
    .filter(ee.Filter.eq("ADM0_NAME", "Germany"))
    .filter(ee.Filter.eq("ADM1_NAME", "Bayern"))
)

n_bavaria = bavaria.size().getInfo()
if n_bavaria == 0:
    raise RuntimeError("Bayern wurde in der GAUL-Grenzdatei nicht gefunden.")

embeddings = ee.ImageCollection(
    "GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL"
)

annual_collection = (
    embeddings
    .filterDate(f"{YEAR}-01-01", f"{YEAR + 1}-01-01")
    .filterBounds(bavaria.geometry())
)

n_images = annual_collection.size().getInfo()

print("Bayern-Features:", n_bavaria)
print(f"AlphaEarth-Bilder für {YEAR}:", n_images)

if n_images == 0:
    raise RuntimeError(
        f"Für {YEAR} wurden keine AlphaEarth-Bilder gefunden. "
        "Bitte ein verfügbares Jahr auswählen."
    )

print("Bänder:", annual_collection.first().bandNames().getInfo())

Bayern-Features: 1
AlphaEarth-Bilder für 2022: 13
Bänder: ['A00', 'A01', 'A02', 'A03', 'A04', 'A05', 'A06', 'A07', 'A08', 'A09', 'A10', 'A11', 'A12', 'A13', 'A14', 'A15', 'A16', 'A17', 'A18', 'A19', 'A20', 'A21', 'A22', 'A23', 'A24', 'A25', 'A26', 'A27', 'A28', 'A29', 'A30', 'A31', 'A32', 'A33', 'A34', 'A35', 'A36', 'A37', 'A38', 'A39', 'A40', 'A41', 'A42', 'A43', 'A44', 'A45', 'A46', 'A47', 'A48', 'A49', 'A50', 'A51', 'A52', 'A53', 'A54', 'A55', 'A56', 'A57', 'A58', 'A59', 'A60', 'A61', 'A62', 'A63']


### Dateigröße der einzelnen GeoTIFFs

Earth Engine berechnet für die 64 AlphaEarth-Bänder derzeit **512 Byte pro Pixel**, also 64 Bänder × 8 Byte (`float64`). Eine Datei mit `8192 × 8192` Pixeln hätte deshalb vor Kompression 32 GiB und überschreitet das zulässige Maximum von 16 GiB.

Mit `4096 × 4096` Pixeln liegt jede Ausgabedatei bei ungefähr 8 GiB vor Kompression. Der gesamte Export eines Jahres bleibt weiterhin etwa 141–142 GB groß, wird aber auf mehr Einzeldateien verteilt.

## 4. Export vorbereiten

Es wird nur das oben ausgewählte Jahr exportiert. `fileDimensions=4096` teilt die Ausgabe in mehrere handhabbare GeoTIFF-Dateien. `skipEmptyTiles=True` verhindert Dateien für Bereiche außerhalb Bayerns.

In [12]:
image = (
    annual_collection
    .mosaic()
    .clip(bavaria.geometry())
)

task = ee.batch.Export.image.toDrive(
    image=image,
    description=f"AlphaEarth_Bayern_{YEAR}",
    folder=DRIVE_FOLDER,
    fileNamePrefix=FILE_PREFIX,
    region=bavaria.geometry(),
    scale=10,
    maxPixels=1e13,

    # Große Ausgabe in mehrere Dateien aufteilen
    shardSize=256,
    fileDimensions=FILE_DIMENSIONS,
    skipEmptyTiles=True,

    fileFormat="GeoTIFF",
    formatOptions={
        "cloudOptimized": True
    }
)

print("Export vorbereitet:")
print("  Jahr:", YEAR)
print("  Beschreibung:", f"AlphaEarth_Bayern_{YEAR}")
print("  Zielordner:", DRIVE_FOLDER)
print("  Erwartete Gesamtgröße: ungefähr 141–142 GB")
print()
print("Die nächste Zelle startet den Export.")

Export vorbereitet:
  Jahr: 2022
  Beschreibung: AlphaEarth_Bayern_2022
  Zielordner: AlphaEarth_Bavaria
  Erwartete Gesamtgröße: ungefähr 141–142 GB

Die nächste Zelle startet den Export.


## 5. Export starten

Diese Zelle nur einmal pro Jahr ausführen. Ein erneutes Ausführen erzeugt einen weiteren Task mit demselben Präfix.

In [13]:
task.start()

status = task.status()

print("Export gestartet.")
print("Task:", status.get("description"))
print("Status:", status.get("state"))
print("Task-ID:", status.get("id"))

Export gestartet.
Task: AlphaEarth_Bayern_2022
Status: READY
Task-ID: 4TZQPHLQQAHIHBTKBKEGH624


## 6. Status überwachen

Die Dateien erscheinen erst in Google Drive, wenn der Task `COMPLETED` erreicht hat. Während `READY` oder `RUNNING` ist normalerweise noch nichts im Drive-Ordner sichtbar.

In [14]:
POLL_INTERVAL_SECONDS = 60

while True:
    status = task.status()
    state = status.get("state", "UNKNOWN")
    now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    print(
        f"[{now}] "
        f"{status.get('description', 'unknown')}: {state}"
    )

    if state in {"COMPLETED", "FAILED", "CANCELLED"}:
        break

    time.sleep(POLL_INTERVAL_SECONDS)

print("-" * 60)
print("Finaler Status:", state)

if state == "COMPLETED":
    print(
        f"Export abgeschlossen. Prüfe den Drive-Ordner "
        f"'{DRIVE_FOLDER}'."
    )
elif state == "FAILED":
    print("Fehlermeldung:")
    print(status.get("error_message", "Keine Fehlermeldung verfügbar."))

[2026-07-22 14:23:27] AlphaEarth_Bayern_2022: READY
[2026-07-22 14:24:27] AlphaEarth_Bayern_2022: RUNNING
[2026-07-22 14:25:28] AlphaEarth_Bayern_2022: RUNNING
[2026-07-22 14:26:28] AlphaEarth_Bayern_2022: RUNNING
[2026-07-22 14:27:28] AlphaEarth_Bayern_2022: RUNNING
[2026-07-22 14:28:29] AlphaEarth_Bayern_2022: RUNNING
[2026-07-22 14:29:29] AlphaEarth_Bayern_2022: RUNNING
[2026-07-22 14:30:30] AlphaEarth_Bayern_2022: RUNNING
[2026-07-22 14:31:30] AlphaEarth_Bayern_2022: RUNNING
[2026-07-22 14:32:30] AlphaEarth_Bayern_2022: RUNNING
[2026-07-22 14:33:30] AlphaEarth_Bayern_2022: RUNNING
[2026-07-22 14:34:31] AlphaEarth_Bayern_2022: RUNNING
[2026-07-22 14:35:32] AlphaEarth_Bayern_2022: RUNNING
[2026-07-22 14:36:32] AlphaEarth_Bayern_2022: RUNNING
[2026-07-22 14:37:33] AlphaEarth_Bayern_2022: RUNNING
[2026-07-22 14:38:33] AlphaEarth_Bayern_2022: RUNNING
[2026-07-22 14:39:34] AlphaEarth_Bayern_2022: RUNNING
[2026-07-22 14:40:34] AlphaEarth_Bayern_2022: RUNNING
[2026-07-22 14:41:34] AlphaEar

## 7. Status später erneut prüfen

Falls der Kernel beendet oder das Notebook geschlossen wurde, kann der Task über seine Task-ID später erneut geprüft werden. Kopiere dafür die beim Start ausgegebene Task-ID in `TASK_ID`.

Alternativ kannst du die Tasks auch direkt im Earth Engine Task Manager kontrollieren.

In [ ]:
# Optional: Status eines bereits gestarteten Tasks abfragen
TASK_ID = ""  # hier bei Bedarf die Task-ID eintragen

if TASK_ID:
    saved_status = ee.data.getTaskStatus(TASK_ID)[0]

    print("Task:", saved_status.get("description"))
    print("Status:", saved_status.get("state"))

    if saved_status.get("error_message"):
        print("Fehler:", saved_status["error_message"])
else:
    print("Keine TASK_ID eingetragen.")

## 8. Nach jedem Jahr

Nach einem erfolgreichen Export:

1. Alle Dateien mit dem Präfix `alphaearth_bayern_<JAHR>` aus dem Drive-Ordner herunterladen.
2. Prüfen, ob der lokale Download vollständig ist.
3. Die Dateien aus Google Drive löschen.
4. Den **Google-Drive-Papierkorb leeren**, da der Speicher sonst weiterhin belegt bleibt.
5. `YEAR` oben auf das nächste Jahr setzen.
6. Ab Zelle 3 erneut ausführen.

Die fehlgeschlagenen alten Tasks belegen keinen Drive-Speicher und müssen nicht gelöscht werden.